# 07 — SAM2.1 ONNX export attempt (honest)

ONNX decoder export is supported for mobilesam / sam-vit-b/l/h. SAM2.1 is not currently eligible — we report that honestly instead of faking a file.

> VisionServeX does **not** redistribute gated or restricted model weights. You bring your own token and accept upstream licenses yourself. Tokens are always redacted.

In [ ]:
# Install the published package from PyPI (run AFTER release).
# Before release you may instead `pip install dist/visionservex-3.8.0-py3-none-any.whl`.
# %pip install -q visionservex==3.8.0
import importlib.metadata as _m
print("installed:", _m.version("visionservex"))

In [ ]:
# Assert we are using the INSTALLED package (site-packages), never the local src.
import visionservex
print("visionservex:", visionservex.__version__)
print("file:", visionservex.__file__)
assert "site-packages" in visionservex.__file__, (
    "This tutorial must run against the pip-installed package, not local src. "
    "Use a fresh venv / clean kernel and install visionservex from PyPI."
)

In [ ]:
from visionservex.onnx_export import onnx_eligible
print("ONNX-eligible:", sorted(onnx_eligible()))
print("sam2.1 eligible:", "sam2.1-hiera-small" in onnx_eligible())

In [ ]:
from visionservex import VSX, VSXError
out = {}
try:
    VSX.sam("sam2.1-hiera-small").to_onnx("sam21.onnx")
    out = {"sam2.1": "exported"}
except VSXError as e:
    out = {"sam2.1": f"not_applicable: {e.state}"}
print(out)

In [ ]:
# Record an execution-ledger row (artifact or honest blocker).
import csv, json, os, time
from pathlib import Path
ART = Path("v38_tutorial_artifacts"); ART.mkdir(exist_ok=True)
def record(notebook, status, detail, artifact=None):
    art = None
    if artifact is not None:
        art = ART / f"{notebook}.json"
        art.write_text(json.dumps(artifact, indent=2, default=str))
    row = {"notebook": notebook, "status": status, "detail": detail,
           "artifact": str(art) if art else "", "version": __import__("visionservex").__version__}
    led = ART / "v38_tutorial_execution_ledger.csv"
    new = not led.exists()
    with led.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new: w.writeheader()
        w.writerow(row)
    print("ledger +", row)

In [ ]:
record("07_onnx", "ok", "honest ONNX eligibility reported", out)